# Study 950 — Zero-Coupon Convexity — the teardown

The rate-factor duration match, the excess-of-cash race, the asymmetry regression in both quadratic specifications, the move-size buckets (raw and linearly hedged), block-bootstrap CIs on the mean and on *b2*, the era cut, three sweeps, the ZROZ cross-check and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `a669055b6e7a`).

**Design.** Arm A = 100% EDV. Arm B = `L`·TLT + `(1−L)`·BIL, `L` = ratio of the rolling 252-day OLS slopes of each leg's daily excess return on `Δ^TYX`, read at month end and traded the following month (**one execution lag**). Both arms excess-of-cash. 3 bps one-way × NAV on the mix's turnover; a **PROXY** 25 bp/yr financing spread on `(L−1)`. Both frictions fall on Arm B only, so the race is tilted *towards* the claim.

In [1]:
R = {'start': '2009-02-02', 'end': '2026-06-30', 'n_days': 4377, 'n_months': 208, 'fp': 'a669055b6e7a', 'L_mean': 1.42, 'L_sd': 0.097, 'L_lo': 1.19, 'L_hi': 1.62, 'a_mean': 2.65, 'a_vol': 21.41, 'a_sharpe': 0.124, 'a_t': 0.57, 'b_mean': 3.19, 'b_vol': 21.42, 'b_sharpe': 0.149, 'b_t': 0.71, 'sp_mean': -0.53, 'sp_vol': 4.77, 'sp_sharpe': -0.213, 'sp_t': -0.87, 'vol_ratio': 1.0, 'sp_bp_mo': -3.98, 'sp_bp_t': -1.08, 'sp_hit': 44.2, 'dy2_a': -8.75, 'dy2_a_t': -1.5, 'dy2_b1': -0.7217, 'dy2_b1_t': -2.73, 'dy2_b2': 110.5, 'dy2_b2_t': 0.84, 'dy2_25': 6.9, 'dy2_50': 27.62, 'dy2_be': 28.1, 'dy2_r2': 0.079, 'rv_a': -12.74, 'rv_a_t': -1.88, 'rv_b1': -0.7108, 'rv_b1_t': -2.55, 'rv_b2': 154.7, 'rv_b2_t': 1.31, 'rv_25': 9.67, 'rv_50': 38.69, 'rv_be': 28.7, 'rv_r2': 0.084, 'median_absdy': 15.0, 'bk_q_n': 70, 'bk_q_dy': 4.1, 'bk_q': -0.94, 'bk_q_t': -0.15, 'bk_q_h': -1.22, 'bk_m_n': 69, 'bk_m_dy': 14.5, 'bk_m': -7.51, 'bk_m_t': -1.22, 'bk_m_h': -7.07, 'bk_l_n': 69, 'bk_l_dy': 32.4, 'bk_l': -3.52, 'bk_l_t': -0.43, 'bk_l_h': -2.38, 'bk_q_pred': 0.27, 'bk_l_pred': 12.93, 'bk_gap_se': 11.6, 'boot_lo': -11.01, 'boot_hi': 3.42, 'boot_neg': 86.8, 'b2_dy2_lo': -163.3, 'b2_dy2_hi': 338.2, 'b2_dy2_neg': 27.5, 'b2_rv_lo': -41.0, 'b2_rv_hi': 516.2, 'b2_rv_neg': 5.9, 'e_n': 106, 'e_sp': -4.21, 'e_sp_t': -0.67, 'e_a': -12.78, 'e_a_t': -1.44, 'e_b2': 179.9, 'e_b2_t': 0.85, 'e_b1_t': -2.06, 'l_n': 101, 'l_sp': -4.17, 'l_sp_t': -1.18, 'l_a': -4.18, 'l_a_t': -0.94, 'l_b2': 21.8, 'l_b2_t': 0.39, 'l_b1_t': -2.41, 'cost0_sp': -4.0, 'cost25_sp': -3.78, 'fin0_sp': -4.85, 'fin0_t': -1.31, 'fin100_sp': -1.35, 'fin100_t': -0.37, 'w126_sp': -0.09, 'w126_b2': 87.1, 'w504_sp': -1.18, 'w504_b2': 98.2, 'z_start': '2010-12-01', 'z_months': 186, 'z_L': 1.549, 'z_volratio': 0.992, 'z_sp': -2.06, 'z_sp_t': -0.46, 'z_b2_dy2': 208.7, 'z_b2_dy2_t': 1.69, 'z_b2_rv': 51.7, 'z_b2_rv_t': 0.27, 'z_a_dy2': -11.43, 'z_a_dy2_t': -1.75, 'z_b1_t': -1.4, 'ze_b2_dy2': 403.8, 'ze_b2_dy2_t': 2.14, 'ze_b2_rv': 618.9, 'ze_b2_rv_t': 2.88, 'ze_a_rv': -33.19, 'zl_b2_rv': -123.5, 'zl_b2_rv_t': -1.81, 'zl_a_rv': 5.09, 'grid_n': 12, 'grid_b2_pos': 11, 'grid_a_neg': 11, 'grid_t2': 2, 'grid_max_t': 2.88, 'edv_max_t': 1.87, 'syn_b2': 260.3, 'syn_b2_t': 6.31, 'syn_a': -27.05, 'syn_a_t': -7.41, 'syn_volratio': 1.002, 'syn_null_t': -0.93, 'syn_null_sd': 1.12, 'syn_null_fire': 1}

## 1. The match — does it actually neutralise duration?

In [2]:
print(f"L on TLT: mean {R['L_mean']:.3f}  sd {R['L_sd']:.3f}  "
      f"range [{R['L_lo']:.2f}, {R['L_hi']:.2f}]")
print(f"A: 100% EDV        mean {R['a_mean']:+.2f}%/yr  vol {R['a_vol']:.2f}%  "
      f"Sharpe {R['a_sharpe']:+.3f}  HAC t {R['a_t']:+.2f}")
print(f"B: matched mix     mean {R['b_mean']:+.2f}%/yr  vol {R['b_vol']:.2f}%  "
      f"Sharpe {R['b_sharpe']:+.3f}  HAC t {R['b_t']:+.2f}")
print(f"A - B (spread)     mean {R['sp_mean']:+.2f}%/yr  vol {R['sp_vol']:.2f}%  "
      f"Sharpe {R['sp_sharpe']:+.3f}  HAC t {R['sp_t']:+.2f}")
print(f"vol ratio A/B = {R['vol_ratio']:.3f}  -> a clean duration match")
print(f"monthly: {R['sp_bp_mo']:+.2f} bp/mo (t {R['sp_bp_t']:+.2f}), "
      f"hit rate {R['sp_hit']:.1f}%, bootstrap 95% CI "
      f"[{R['boot_lo']:+.2f}, {R['boot_hi']:+.2f}] bp/mo, share<0 {R['boot_neg']:.1f}%")

L on TLT: mean 1.420  sd 0.097  range [1.19, 1.62]
A: 100% EDV        mean +2.65%/yr  vol 21.41%  Sharpe +0.124  HAC t +0.57
B: matched mix     mean +3.19%/yr  vol 21.42%  Sharpe +0.149  HAC t +0.71
A - B (spread)     mean -0.53%/yr  vol 4.77%  Sharpe -0.213  HAC t -0.87
vol ratio A/B = 1.000  -> a clean duration match
monthly: -3.98 bp/mo (t -1.08), hit rate 44.2%, bootstrap 95% CI [-11.01, +3.42] bp/mo, share<0 86.8%


> 💡 **In plain words** — the two arms end up with the *same* interest-rate risk (volatilities 21.41% vs 21.42%). Anything left over is not duration. Over the whole sample that leftover is slightly negative and statistically nothing.

## 2. The headline — `diff = a + b1·Δy + b2·Q`, HAC(6)

`Q` is either the squared net monthly move `Δy²` (a bond repriced from one month-end yield to the next) or the realised variance `Σ Δy_t²` of the daily factor inside the month (the gamma P&L a daily-marked fund actually accrues). Theory: `b2 > 0`, `a < 0`, `b1 ≈ 0`.

In [3]:
for tag, a, at, b1, b1t, b2, b2t, p25, p50, be, r2 in [
    ('dy^2 (net move) ', R['dy2_a'], R['dy2_a_t'], R['dy2_b1'], R['dy2_b1_t'],
     R['dy2_b2'], R['dy2_b2_t'], R['dy2_25'], R['dy2_50'], R['dy2_be'], R['dy2_r2']),
    ('realised variance', R['rv_a'], R['rv_a_t'], R['rv_b1'], R['rv_b1_t'],
     R['rv_b2'], R['rv_b2_t'], R['rv_25'], R['rv_50'], R['rv_be'], R['rv_r2'])]:
    print(f"[{tag}]  R2={r2:.3f}")
    print(f"   a  = {a:+7.2f} bp/mo (t {at:+.2f})   b1 = {b1:+.4f} (t {b1t:+.2f}) "
          f"-> residual duration {-b1:+.2f} yr")
    print(f"   b2 = {b2:+7.1f}        (t {b2t:+.2f})   worth {p25:+.2f} bp at 25 bp, "
          f"{p50:+.2f} bp at 50 bp; breakeven move {be:.1f} bp")
print(f"\nmedian |monthly move| = {R['median_absdy']:.1f} bp "
      f"-> the breakeven sits at roughly the 67th percentile of months")
print(f"bootstrap b2 [dy^2]: 95% CI [{R['b2_dy2_lo']:+.1f}, {R['b2_dy2_hi']:+.1f}], "
      f"share<0 {R['b2_dy2_neg']:.1f}%")
print(f"bootstrap b2 [rv  ]: 95% CI [{R['b2_rv_lo']:+.1f}, {R['b2_rv_hi']:+.1f}], "
      f"share<0 {R['b2_rv_neg']:.1f}%")

[dy^2 (net move) ]  R2=0.079
   a  =   -8.75 bp/mo (t -1.50)   b1 = -0.7217 (t -2.73) -> residual duration +0.72 yr
   b2 =  +110.5        (t +0.84)   worth +6.90 bp at 25 bp, +27.62 bp at 50 bp; breakeven move 28.1 bp
[realised variance]  R2=0.084
   a  =  -12.74 bp/mo (t -1.88)   b1 = -0.7108 (t -2.55) -> residual duration +0.71 yr
   b2 =  +154.7        (t +1.31)   worth +9.67 bp at 25 bp, +38.69 bp at 50 bp; breakeven move 28.7 bp

median |monthly move| = 15.0 bp -> the breakeven sits at roughly the 67th percentile of months
bootstrap b2 [dy^2]: 95% CI [-163.3, +338.2], share<0 27.5%
bootstrap b2 [rv  ]: 95% CI [-41.0, +516.2], share<0 5.9%


> 💡 **In plain words** — the curvature term points the right way and is about the right size for a 7.5-year duration gap. It is simply drowned in noise: a 25 bp month is supposed to hand you ~7 bp, and the month-to-month scatter is ~140 bp.

## 3. The same thing without a model — move-size buckets

Terciles of |Δy|, raw and after regressing out the residual linear exposure (in-sample, descriptive — used to look at *shape*, never to claim a return).

In [4]:
print(f"{'bucket':10s}{'n':>5s}{'|dy|':>10s}{'spread':>12s}{'t':>8s}{'hedged':>12s}")
for nm, n, dy, sp, t, h in [
    ('quiet', R['bk_q_n'], R['bk_q_dy'], R['bk_q'], R['bk_q_t'], R['bk_q_h']),
    ('middling', R['bk_m_n'], R['bk_m_dy'], R['bk_m'], R['bk_m_t'], R['bk_m_h']),
    ('large', R['bk_l_n'], R['bk_l_dy'], R['bk_l'], R['bk_l_t'], R['bk_l_h'])]:
    print(f"{nm:10s}{n:5d}{dy:9.1f}bp{sp:+11.2f}bp{t:+8.2f}{h:+11.2f}bp")
print('\nno monotone climb -> the raw asymmetry is absent.')
gap = R['bk_l_pred'] - R['bk_q_pred']
print(f"consistency check: at b2 = {R['dy2_b2']:+.0f} the fit predicts "
      f"{R['bk_q_pred']:+.2f} bp in the quiet bucket and {R['bk_l_pred']:+.2f} bp "
      f"in the large one")
print(f"  -> a gap of {gap:.1f} bp/mo against a standard error on that gap of "
      f"{R['bk_gap_se']:.1f} bp: about ONE standard error.")
print('  The bucket table and the regression do not disagree - the effect is simply')
print('  at the edge of what 208 months can resolve.')

bucket        n      |dy|      spread       t      hedged
quiet        70      4.1bp      -0.94bp   -0.15      -1.22bp
middling     69     14.5bp      -7.51bp   -1.22      -7.07bp
large        69     32.4bp      -3.52bp   -0.43      -2.38bp

no monotone climb -> the raw asymmetry is absent.
consistency check: at b2 = +110 the fit predicts +0.27 bp in the quiet bucket and +12.93 bp in the large one
  -> a gap of 12.7 bp/mo against a standard error on that gap of 11.6 bp: about ONE standard error.
  The bucket table and the regression do not disagree - the effect is simply
  at the edge of what 208 months can resolve.


## 4. Era cut (split 2018-01-01)

In [5]:
print(f"2009-2017 (n={R['e_n']:3d} mo): spread {R['e_sp']:+.2f} bp/mo (t {R['e_sp_t']:+.2f})  "
      f"a {R['e_a']:+.2f} (t {R['e_a_t']:+.2f})  b2 {R['e_b2']:+7.1f} (t {R['e_b2_t']:+.2f})  "
      f"b1 t {R['e_b1_t']:+.2f}")
print(f"2018-2026 (n={R['l_n']:3d} mo): spread {R['l_sp']:+.2f} bp/mo (t {R['l_sp_t']:+.2f})  "
      f"a {R['l_a']:+.2f} (t {R['l_a_t']:+.2f})  b2 {R['l_b2']:+7.1f} (t {R['l_b2_t']:+.2f})  "
      f"b1 t {R['l_b1_t']:+.2f}")
print('\nb2 keeps its sign but collapses 8x into the era that CONTAINS 2020 and 2022 -')
print('the two biggest rate shocks of the sample. The only era-stable feature is the')
print('residual duration leak (b1 t = %+.2f / %+.2f).' % (R['e_b1_t'], R['l_b1_t']))

2009-2017 (n=106 mo): spread -4.21 bp/mo (t -0.67)  a -12.78 (t -1.44)  b2  +179.9 (t +0.85)  b1 t -2.06
2018-2026 (n=101 mo): spread -4.17 bp/mo (t -1.18)  a -4.18 (t -0.94)  b2   +21.8 (t +0.39)  b1 t -2.41

b2 keeps its sign but collapses 8x into the era that CONTAINS 2020 and 2022 -
the two biggest rate shocks of the sample. The only era-stable feature is the
residual duration leak (b1 t = -2.06 / -2.41).


## 5. Sweeps — costs, the financing PROXY, and the hedge lookback

In [6]:
print(f"cost   0 bps: spread {R['cost0_sp']:+.2f} bp/mo   |   "
      f"cost  25 bps: spread {R['cost25_sp']:+.2f} bp/mo   (turnover is a rounding error)")
print(f"financing  0 bp/yr: {R['fin0_sp']:+.2f} bp/mo (t {R['fin0_t']:+.2f})   |   "
      f"100 bp/yr: {R['fin100_sp']:+.2f} bp/mo (t {R['fin100_t']:+.2f})")
print('   ^ the PROXY. A WIDER spread makes the mix worse and FLATTERS the zero leg -')
print('     even at a punitive 100 bp/yr the zero still does not get ahead.')
print(f"window 126d: spread {R['w126_sp']:+.2f} bp/mo, b2 {R['w126_b2']:+.1f}   |   "
      f"252d: {R['sp_bp_mo']:+.2f}, b2 {R['dy2_b2']:+.1f}   |   "
      f"504d: {R['w504_sp']:+.2f}, b2 {R['w504_b2']:+.1f}")

cost   0 bps: spread -4.00 bp/mo   |   cost  25 bps: spread -3.78 bp/mo   (turnover is a rounding error)
financing  0 bp/yr: -4.85 bp/mo (t -1.31)   |   100 bp/yr: -1.35 bp/mo (t -0.37)
   ^ the PROXY. A WIDER spread makes the mix worse and FLATTERS the zero leg -
     even at a punitive 100 bp/yr the zero still does not get ahead.
window 126d: spread -0.09 bp/mo, b2 +87.1   |   252d: -3.98, b2 +110.5   |   504d: -1.18, b2 +98.2


> 💡 **In plain words** — nothing about the plumbing decides this study. Change the costs, change the borrowing assumption, change how the hedge is estimated: the curvature term stays positive and stays insignificant, and the mean spread wobbles around zero. That *is* the finding.

## 6. Cross-check — ZROZ (25y+ zeros)

In [7]:
print(f"{R['z_start']} -> 2026-06-30, {R['z_months']} months, L {R['z_L']:.3f}, "
      f"vol ratio {R['z_volratio']:.3f}")
print(f"spread {R['z_sp']:+.2f} bp/mo (t {R['z_sp_t']:+.2f})")
print(f"b2[dy^2] {R['z_b2_dy2']:+.1f} (t {R['z_b2_dy2_t']:+.2f})   "
      f"b2[rv] {R['z_b2_rv']:+.1f} (t {R['z_b2_rv_t']:+.2f})  <- a factor of 4 apart")
print(f"a[dy^2] {R['z_a_dy2']:+.2f} bp/mo (t {R['z_a_dy2_t']:+.2f}); "
      f"b1 t {R['z_b1_t']:+.2f} (leak milder - ZROZ sits closer to the 30y point)")
print('\nsame signs, same absence of significance on the full sample, and an estimate')
print('that is not stable across the two quadratic specifications. Nothing to bank.')

2010-12-01 -> 2026-06-30, 186 months, L 1.549, vol ratio 0.992
spread -2.06 bp/mo (t -0.46)
b2[dy^2] +208.7 (t +1.69)   b2[rv] +51.7 (t +0.27)  <- a factor of 4 apart
a[dy^2] -11.43 bp/mo (t -1.75); b1 t -1.40 (leak milder - ZROZ sits closer to the 30y point)

same signs, same absence of significance on the full sample, and an estimate
that is not stable across the two quadratic specifications. Nothing to bank.


## 6b. The full cut census — and the only two cuts that clear |*t*| = 2

A robustness claim is only checkable if **every** cut the design implies is on the page. The design implies **12**: two funds × three eras × two quadratic specifications. `strategy.cut_grid` / `grid_census` run and print all of them (`examples/verify.py`), so no sentence in this study can quote a favourable subset.

An earlier draft reported six of them, called the sign pattern unanimous, and said the best *t* anywhere was +1.69. Fitting the six it had not run — the ZROZ era cut — overturned both statements.

In [8]:
census = dict(n=12, b2_pos=11, a_neg=11,
              t2=2, max_t=2.88, edv_max_t=1.87)
print('census over the full grid of %d cuts:' % census['n'])
print('  b2 > 0 in %2d/%d   a < 0 in %2d/%d   |t| >= 2 in %d/%d'
      % (census['b2_pos'], census['n'], census['a_neg'], census['n'],
         census['t2'], census['n']))
print('  largest |t| ANYWHERE: %+.2f  (ZROZ, 2010-2017, realised variance)'
      % census['max_t'])
print('  the two cuts that clear 2: ZROZ 2010-2017  b2 +403.8 (t +2.14) [dy^2]')
print('%38s b2 +618.9 (t +2.88) [rv]' % '')
print('  ... and the era NEXT DOOR, same fund, same spec:')
print('     ZROZ 2018-2026  b2 -123.5 (t -1.81), a +5.09 bp/mo  <- BOTH SIGNS FLIP')
print('  EDV, the headline fund, never exceeds +1.87 in any cut.')

census over the full grid of 12 cuts:
  b2 > 0 in 11/12   a < 0 in 11/12   |t| >= 2 in 2/12
  largest |t| ANYWHERE: +2.88  (ZROZ, 2010-2017, realised variance)
  the two cuts that clear 2: ZROZ 2010-2017  b2 +403.8 (t +2.14) [dy^2]
                                       b2 +618.9 (t +2.88) [rv]
  ... and the era NEXT DOOR, same fund, same spec:
     ZROZ 2018-2026  b2 -123.5 (t -1.81), a +5.09 bp/mo  <- BOTH SIGNS FLIP
  EDV, the headline fund, never exceeds +1.87 in any cut.


> 💡 **How to read those two cuts.** They are not two findings: they are one 84-month window of the *cross-check* fund, fit twice with two near-identical regressors, un-corrected for 12 looks — roughly the rate at which a null throws up a |*t*| ≥ 2 by chance. The decisive fact is the neighbour: on 2018-2026, the era that actually contains 2020 and 2022, the same fund and the same specification produce `b2` = **-123.5** with a **positive** intercept — convexity running backwards. Reported, not promoted: this is what `WEAK` ("fragile to method or selection") is for, and it is why the badge did not move up.

## 7. The synthetic control — is the harness unbiased? (live, offline)

Planted world: the long leg carries 2.5× the short leg's convexity per unit of duration and pays 30 bp/month of carry for it — `b2` must come out strongly positive and `a` strongly negative. Null world: the duration-matched mix is convexity-matched too — both must be silent. This proves the machinery; it never supports a real-tape stamp.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from zero_convexity import data, strategy as st
pl = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=950)[0])
print(f"planted: b2 {pl['b2']:+.1f} (t {pl['b2_t']:+.2f})  a {pl['a_bp_mo']:+.2f} bp/mo "
      f"(t {pl['a_t']:+.2f})  b1 t {pl['b1_t']:+.2f}  vol ratio {pl['vol_ratio']:.3f}")
ts = np.array([st.synthetic_detect(
        data.synthetic_panel(signal_strength=0.0, seed=950+s)[0])['b2_t']
      for s in range(8)])
print(f"null x8: b2 t mean {ts.mean():+.2f} (sd {ts.std(ddof=1):.2f}), "
      f"|t|>=2 in {(np.abs(ts)>=2).sum()}/8")
print('\nplanted effect resolved at t>5; null centred on zero -> the real-tape silence')
print('is a property of the Treasury curve, not of the regression.')

planted: b2 +260.3 (t +6.31)  a -27.05 bp/mo (t -7.41)  b1 t +0.19  vol ratio 1.002


null x8: b2 t mean -0.93 (sd 1.12), |t|>=2 in 1/8

planted effect resolved at t>5; null centred on zero -> the real-tape silence
is a property of the Treasury curve, not of the regression.


## Verdict

- **Signal — Weak.** The convexity coefficient is positive in **11 of 12** fund × era × specification cuts and the intercept negative in 11 of 12 — the sign pattern of "convexity is real but priced", with one cut (ZROZ 2018-2026, realised variance) flipping both. The headline does not clear the bar: `b2` = **+110.5** (HAC *t* = **+0.84**), **+154.7** (*t* = +1.31) on realised variance; the **2 cuts that do** clear it (ZROZ 2010-2017, **+2.14** / **+2.88**) are one window of the cross-check fund, uncorrected for 12 looks, inverted by the era beside it; bootstrap CIs [-163, +338] and [-41, +516]; the raw large-move bucket is **-3.52 bp/mo**, i.e. the wrong sign; and a significant **+0.72 yr** residual duration (*t* = -2.73, era-stable) means part of the spread is a 20s-30s curve trade. The synthetic control resolves a planted pickup at *t* = +6.31 and is silent on the null (1/8), so the miss is the tape's, not the harness's. **Survivorship / selection:** four named, still-listed funds and one official yield series, but EDV and ZROZ are the *only* US zero-coupon Treasury ETFs that survived to be testable.
- **Tradability — Mirage.** The spread earns **-0.53%/yr** (monthly Sharpe -0.213, *t* -0.87); the fitted convexity needs a **28 bp** month to repay its own carry against a median month of **15 bp**; the estimate does not survive the change of quadratic regressor on ZROZ (+209 → +52). Costs and the financing proxy are immaterial. The zero fund buys you **more duration per dollar** — a real and useful property — and nothing else you can measure.